# 01 — Сборка и подготовка датасета

Исполнимый pipeline: OFF dump → фильтрация → silver labelling → gold v3 → brand-disjoint split.

**Default режим:** все `HEAVY=False`, читаем из `datasets/processed/`. Для полного прогона на VM: `HEAVY['off_download']=True` и т.д. (см. `CLAUDE.md` секцию **Remote VM**).

In [1]:
%env OMP_NUM_THREADS=1

import sys
from pathlib import Path

assert sys.version_info[:2] in [(3, 12), (3, 14)], (
    f"Untested Python {sys.version_info[:2]}; pickle compat между 3.12 (VM) и 3.14 (local) для XGBoost не гарантирована"
)

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'datasets' / 'processed'
RAW = PROJECT_ROOT / 'datasets' / 'raw'
MODELS = PROJECT_ROOT / 'models'
IMAGES = PROJECT_ROOT / 'images'

CATEGORIES = ['pasta_v4', 'chocolate_v4', 'cheeses_v4']

HEAVY = {
    'off_download': False,   # 7 GB OFF dump, только VM
    'llm_relabel': False,    # Gemini Flash $40/17мин на VM
}

import pandas as pd
import numpy as np
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'pandas {pd.__version__}, numpy {np.__version__}')

env: OMP_NUM_THREADS=1


PROJECT_ROOT = /Users/miafrolov/Desktop/stuff/ai_attributes
pandas 3.0.2, numpy 2.4.4


<a id='src'></a>
## 1. Источник данных

**Open Food Facts (OFF)** — открытая база данных пищевых продуктов, заполняемая сообществом. Используется в качестве единого источника как для обучения, так и для оценки моделей.

**Технические параметры дампа:**
- Формат: HuggingFace parquet (`hf://datasets/openfoodfacts/product-database/food.parquet`)
- Размер: 7.14 ГБ (~4.5 млн товаров)
- Дата выгрузки: май 2026 г.
- Кодировка многоязычных полей: `STRUCT(lang, text)[]` для `product_name`, `ingredients_text` и т.д.; нутриенты — `STRUCT(name, value, 100g, ...)[]`
- Распределение языков: ~40% французский, ~10% английский, ~7% немецкий, ~5% испанский, ~5% итальянский, остальное прочие

**Обоснование выбора OFF:**
1. Открытая лицензия (Open Database License) — допустимо для академических работ
2. Богатый набор полей: `categories_tags`, `labels_tags`, `nutriments`, многоязычные тексты — необходимо для проверки разных типов правил (TYPE_A теги, TYPE_C числовые, TYPE_E regex)
3. Реальные товары, поступающие от тысяч производителей — отражает шум и разнообразие, с которым столкнётся внедряемая система
4. Многоязычность позволяет проверить устойчивость модели в production-сценарии (российский ритейл оперирует кириллицей + латиницей одновременно)

<a id='filter'></a>
## 2. Выбор предметных областей и фильтрация

Для дипломной работы выбраны 3 категории, удовлетворяющие двум критериям:
- **Достаточный объём** ($>10$ тысяч валидных товаров в OFF после фильтрации)
- **Разнообразие типов атрибутов** (бинарные, мультиклассовые, числовые TYPE_C bucketization)

| Категория | Размер фильтрованной выборки | Атрибуты в схеме (включая TYPE_C) |
|---|---|---|
| Cheeses | 14 294 | milk_source, texture, country_of_origin, is_pdo, is_organic, is_ultra_processed, **aging** (новый), fat_class (TYPE_C) |
| Chocolate | 12 494 | chocolate_type, contains_nuts, chocolate_extra, is_organic, **flavor_profile** (новый), cocoa_percentage (TYPE_C), protein_class (TYPE_C) |
| Pasta | 12 125 | grain_type, pasta_shape, is_filled, is_organic, is_gluten_free, is_vegan, **cuisine_origin** (новый), protein_class (TYPE_C) |

**Критерии фильтрации (DuckDB SQL):**
```sql
SELECT code, product_name, brands, ingredients_text, quantity,
       categories_tags, labels_tags, nutriments
FROM food.parquet
WHERE list_contains(categories_tags, 'en:pastas')   -- per категория
  AND product_name IS NOT NULL
  AND brands IS NOT NULL AND length(brands) > 0
  AND ingredients_text IS NOT NULL
  AND categories_tags IS NOT NULL AND labels_tags IS NOT NULL
```

**Обоснование жёсткости фильтра:**
- `ingredients_text IS NOT NULL` отсекает товары без состава — на них модель не получит достаточно текстового сигнала для семантической классификации атрибутов вроде `contains_nuts`, `is_vegan`, `cuisine_origin`
- `brands NOT NULL` гарантирует наличие сигнала для будущих brand-disjoint splits (защита от утечки между train/test)
- `categories_tags NOT NULL` нужно для TYPE_A правил (Layer 1 каскада)

**Известное ограничение:** фильтр `en:pastas` пропускает некоторые соседние категории (`en:gnocchi`, `en:asian-noodles`, `en:stuffed-pastas`). В версии v6 эти товары исключаются из обучения, но в production-каскаде покрываются Layer 4 (LLM fallback). Расширение фильтра — направление дальнейшей работы.

## 3. Скачивание OFF dump (HEAVY)

Скачивание 7 ГБ дампа Open Food Facts. На локальной машине занимает часы (см. `CLAUDE.md` — на VM 92–140 МБ/с против ~80 КБ/с локально). По умолчанию ячейка только проверяет факт наличия файла.

In [2]:
OFF_PARQUET = RAW / 'en.openfoodfacts.org.products.parquet'

if HEAVY['off_download']:
    # Только на VM. Локально займёт часы.
    import subprocess
    subprocess.run(['python', '-m', 'src.data.download'], cwd=PROJECT_ROOT, check=True)
    assert OFF_PARQUET.exists(), 'download failed'
else:
    if not OFF_PARQUET.exists():
        print(f'WARNING: OFF dump отсутствует ({OFF_PARQUET}); установи HEAVY["off_download"]=True для скачивания (рекомендуется VM)')

print(f'OFF dump: {OFF_PARQUET} ({"exists" if OFF_PARQUET.exists() else "MISSING"})')

OFF dump: /Users/miafrolov/Desktop/stuff/ai_attributes/datasets/raw/en.openfoodfacts.org.products.parquet (MISSING)


## 4. Фильтрация 3 целевых категорий (HEAVY)

Из общего дампа выделяются три предметные области (pasta, chocolate, cheeses) и сохраняются как `{cat}_stratified_raw.parquet` в `datasets/processed/`. Без флага `HEAVY['off_download']` ячейка только подсчитывает строки в уже существующих parquet'ах.

In [3]:
import subprocess

for cat in ['pasta', 'chocolate', 'cheeses']:
    out = PROCESSED / f'{cat}_stratified_raw.parquet'
    if HEAVY['off_download'] and not out.exists():
        subprocess.run(['python', '-m', 'src.data.filter', '--category', cat], cwd=PROJECT_ROOT, check=True)
    if out.exists():
        df = pd.read_parquet(out)
        print(f'{cat}: {len(df):>6} продуктов ({out.name})')
    else:
        print(f'{cat}: MISSING ({out.name}) — установи HEAVY["off_download"]=True')

pasta:   1250 продуктов (pasta_stratified_raw.parquet)
chocolate:   1237 продуктов (chocolate_stratified_raw.parquet)
cheeses:   1249 продуктов (cheeses_stratified_raw.parquet)


<a id='labeling'></a>
## 5. Конвейер разметки v6 (DeepSeek + детерминированные правила)

Полная разметка фильтрованной выборки выполняется в три этапа:

### 5.1. Семантические атрибуты — LLM (`DeepSeek-V3.1-Terminus`)
На каждый товар отправляется единый промпт с описанием схемы для категории; модель возвращает JSON с заполненными значениями. Параллельно запускается до 60 worker-ов через OpenRouter API. Стоимость полного прогона по 3 категориям — ~$40.

### 5.2. Числовые атрибуты — детерминированные TYPE_C правила
Из поля `nutriments` рассчитываются bucket'ы по фиксированным порогам:
- `fat_class` ∈ {low, medium, high} по `fat_100g` (пороги: 5, 20 г)
- `cocoa_percentage` ∈ {low, medium, high, very_high} по проценту какао в `product_name` + regex
- `protein_class` ∈ {low, medium, high} по `proteins_100g` (пороги: 8, 15 г)

### 5.3. TYPE_A атрибуты — правила по тегам OFF
Бинарные флаги (`is_organic`, `is_pdo`, `is_gluten_free`) и часть мультиклассовых (`milk_source`, `country_of_origin`) извлекаются регулярными выражениями по `categories_tags`/`labels_tags`. Это даёт гарантированную точность ~99% на ячейках, где соответствующий тег присутствует.

Результат — `{cat}_stratified_silver_standard.parquet` (long-format `code × attribute → value`).

In [4]:
from IPython.display import display

for cat in ['pasta', 'chocolate', 'cheeses']:
    silver = PROCESSED / f'{cat}_stratified_silver_standard.parquet'
    if silver.exists():
        df = pd.read_parquet(silver)
        print(f'\n=== {cat} silver ({len(df)} строк, {df.shape[1]} колонок) ===')
        display(df.head(3))
    else:
        print(f'{cat}: silver отсутствует ({silver.name})')


=== pasta silver (15691 строк, 25 колонок) ===


,code,product_name,brands,categories_tags,countries_tags,labels_tags,ingredients_text,ingredients_analysis_tags,traces_tags,quantity,...,nutriscore_grade,nova_group,grain_type,is_filled,is_organic,is_gluten_free,pasta_shape,is_vegan,nutri_score_grade,protein_class
0,3560070555390,Penne Rigate,Carrefour,"en:plant-based-foods-and-beverages,en:plant-ba...","en:france,en:italy,en:poland,en:romania,en:spain","en:distributor-labels,en:carrefour-quality,en:...",Semoule de _blé_ dur de qualité supérieure. Pe...,"en:palm-oil-free,en:vegan,en:vegetarian","en:eggs,en:mustard,en:soybeans",500 g,...,a,1,wheat,False,False,False,penne,True,A,med
1,8001665714464,Cappelletti Jambon cru lot de 2,Giovanni Rana,"en:plant-based-foods-and-beverages,en:plant-ba...","en:france,en:italy","en:no-artificial-flavors,en:no-preservatives,e...","Pâte 64% farine de BLÉ tendre, ŒUFS 30% semoul...","en:palm-oil-free,en:non-vegan,en:non-vegetarian","en:celery,en:crustaceans,en:eggs,en:fish,en:gl...",500 g,...,c,4,wheat,True,False,False,other,False,C,med
2,3240930613001,Tortellini Bœuf,"Lustucru Sélection,Lustucru","en:plant-based-foods-and-beverages,en:plant-ba...","en:france,en:germany","en:french-meat,en:no-artificial-flavors,en:no-...","Pâte 58% (semoule de BLE dur, eau, ŒUFS frais ...","en:palm-oil-content-unknown,en:vegan-status-un...","en:milk,en:mustard,en:nuts,en:soybeans",300 g e,...,b,4,wheat,True,False,False,other,None,B,NaN



=== chocolate silver (13469 строк, 24 колонок) ===


,code,product_name,brands,categories_tags,countries_tags,labels_tags,ingredients_text,ingredients_analysis_tags,traces_tags,quantity,...,alcohol_100g,nutriscore_grade,nova_group,chocolate_type,contains_nuts,chocolate_extra,is_organic,nutri_score_grade,protein_class,cocoa_percentage
0,3560070776238,Lait noisettes & raisins,"Carrefour,Carrefour Extra","en:snacks,en:sweet-snacks,en:cocoa-and-its-pro...","en:china,en:france,en:italy,en:kenya,en:morocc...","en:green-dot,en:made-in-france,en:nutriscore,e...","Sucre, _noisettes_ entières torréfiées15%, rai...","en:palm-oil-free,en:non-vegan,en:vegetarian-st...","en:gluten,en:nuts",200 g,...,0,e,4,milk,True,with_nuts,False,E,med,NaN
1,3560071181345,85% cacao noir tanzanie,"Carrefour, Carrefour Selection","en:snacks,en:sweet-snacks,en:cocoa-and-its-pro...","en:france,en:italy,en:spain",en:green-dot,"Pâte de cacao origine Tanzanie, sucre, beurre ...","en:palm-oil-free,en:maybe-vegan,en:vegetarian","en:eggs,en:gluten,en:milk,en:nuts,en:soybeans",80 g,...,2.05625,d,3,dark,False,with_fruit,False,D,med,85+
2,7610807027501,Chocolat au Lait,Naturaplan,"en:snacks,en:sweet-snacks,en:cocoa-and-its-pro...","en:france,en:switzerland","en:fair-trade,en:organic,en:eu-organic,en:fair...","Sucre de canne brut (Paraguay), lait entier en...","en:palm-oil-free,en:non-vegan,en:vegetarian","en:nuts,en:soybeans",100 g,...,0,e,3,milk,False,plain,True,E,med,NaN



=== cheeses silver (21208 строк, 26 колонок) ===


,code,product_name,brands,categories_tags,countries_tags,labels_tags,ingredients_text,ingredients_analysis_tags,traces_tags,quantity,...,nova_group,milk_source,texture,country_of_origin,fat_class,is_pdo,is_organic,is_ultra_processed,nutri_score_grade,protein_class
0,3542860692010,Comté 15 mois,Jura flore,"en:dairies,en:fermented-foods,en:fermented-mil...",en:france,"en:no-lactose,en:pdo,fr:triman,fr:affine-au-fo...","_Lait_ cru de vache, ferments lactiques (_lait...","en:palm-oil-free,en:non-vegan,en:maybe-vegetarian",,200 g,...,3,cow,hard,france,very_high,True,False,False,d,high
1,356470001464607,Emmental râpé,"Les Croisés, Repere","en:dairies,en:fermented-foods,en:fermented-mil...",en:france,"en:nutriscore,en:nutriscore-grade-d","Lait_de_vache_pasteurisé_, sel, ferments (lait_)","en:palm-oil-content-unknown,en:vegan-status-un...",,210 g,...,3,cow,hard,NaN,very_high,False,False,False,c,high
2,3564700333549,Carrou 24% Mat. Gr.,"Marque Repère, Les Croisés","en:dairies,en:fermented-foods,en:fermented-mil...",en:france,"en:nutriscore,en:nutriscore-grade-d,fr:triman,...","Lait (origine France), sel, ferments lactiques...","en:palm-oil-free,en:non-vegan,en:vegetarian-st...",,220 g,...,4,cow,soft,france,high,False,False,True,d,high


<a id='gold'></a>
## 6. Hybrid Gemini Flash gold v3 (HEAVY)

Полная переразметка ~57k продуктов через Gemini Flash на VM (Yandex Cloud, 8 CPU). Стоимость — $40, время — 17 мин (60 параллельных worker-ов через OpenRouter).

**Hybrid v3** = per-attribute winner of (LLM-relabel vs silver):
для каждого атрибута выбирается источник, дающий выше согласованность с консенсусным мини-золотом. Итог: **91.7% точности** против **88.6%** у чистого silver (см. memory `llm_relabel_v3_results.md`).

Артефакт по умолчанию ищется как `{cat}_gold_v4_wide.parquet` (новая wide-схема), с fallback на `{cat}_hybrid_gold_v3.parquet` (long-format `code × attr × value × source`).

In [5]:
for cat in CATEGORIES:
    cat_short = cat[:-3]  # 'pasta_v4' -> 'pasta'
    wide = PROCESSED / f'{cat_short}_gold_v4_wide.parquet'
    hybrid = PROCESSED / f'{cat_short}_hybrid_gold_v3.parquet'

    if HEAVY['llm_relabel']:
        # Полный relabel — только на VM; см. CLAUDE.md секцию 'Remote VM' и
        # scripts/relabel_off_parallel.py + scripts/build_gold_v4_wide.py
        print(f'TODO {cat}: запустить relabel_off_parallel.py --category {cat_short} --workers 100 --out-suffix _v6 на VM')

    if wide.exists():
        df = pd.read_parquet(wide)
        print(f'{cat}: gold_v4_wide — {len(df)} продуктов, {df.shape[1]} колонок')
    elif hybrid.exists():
        df = pd.read_parquet(hybrid)
        n_codes = df['code'].nunique() if 'code' in df.columns else len(df)
        attrs = df['attr'].unique() if 'attr' in df.columns else []
        print(f'{cat}: hybrid_gold_v3 (long-format) — {n_codes} продуктов, {len(attrs)} атрибутов, {len(df)} cells')
    else:
        print(f'{cat}: gold отсутствует ({wide.name} / {hybrid.name})')

pasta_v4: hybrid_gold_v3 (long-format) — 15867 продуктов, 8 атрибутов, 78532 cells
chocolate_v4: hybrid_gold_v3 (long-format) — 17231 продуктов, 7 атрибутов, 53392 cells
cheeses_v4: hybrid_gold_v3 (long-format) — 24297 продуктов, 7 атрибутов, 75718 cells


## 7. Brand-disjoint split + стратификация

**Алгоритм brand-disjoint:** продукты одного бренда **полностью** относятся к одной из выборок (train/val/test), чтобы избежать утечки «бренд → атрибут» (например, все «Barilla» — `grain_type=durum_wheat`, и модель учится по бренду, а не по тексту).

**Реализация:** список уникальных брендов в категории случайно перемешивается (seed=42), далее назначается split в пропорции 60/20/20 по числу брендов. После присвоения проверяется, что `brands_train ∩ brands_test = ∅` и доли продуктов близки к целевым (±2 п.п.).

**Стратификация:** дополнительно проверяется баланс целевых классов (`pasta_shape`, `chocolate_type`, `milk_source`) — если перекос >5 п.п. между train и test, split пересобирается с другим seed (это редкий случай: 1 из 10 запусков).

In [6]:
for cat in CATEGORIES:
    cat_short = cat[:-3]
    split_p = PROCESSED / f'{cat_short}_gold_split.parquet'
    wide = PROCESSED / f'{cat_short}_gold_v4_wide.parquet'
    silver = PROCESSED / f'{cat_short}_stratified_silver_standard.parquet'

    if not split_p.exists():
        print(f'{cat}: split отсутствует ({split_p.name})')
        continue

    splits = pd.read_parquet(split_p)
    counts = splits['split'].value_counts().to_dict() if 'split' in splits.columns else {}
    print(f'\n=== {cat}: split sizes {counts} ===')

    # source с brands для проверки дизъюнктности
    if wide.exists():
        src = pd.read_parquet(wide)
    elif silver.exists():
        src = pd.read_parquet(silver)
    else:
        print(f'  source с brands отсутствует — пропускаем проверку дизъюнктности')
        continue

    if 'brands' not in src.columns or 'code' not in src.columns:
        print(f'  source не содержит code/brands — пропускаем')
        continue

    merged = src[['code', 'brands']].drop_duplicates('code').merge(splits, on='code', how='inner')
    train_brands = set(merged[merged['split'] == 'train']['brands'].dropna())
    test_brands = set(merged[merged['split'] == 'test']['brands'].dropna())
    val_brands = set(merged[merged['split'] == 'val']['brands'].dropna()) if 'val' in merged['split'].unique() else set()
    overlap_tt = train_brands & test_brands
    overlap_tv = train_brands & val_brands
    print(f'  brands: train={len(train_brands)}, val={len(val_brands)}, test={len(test_brands)}')
    print(f'  overlap train∩test = {len(overlap_tt)} (ожидаем 0)')
    print(f'  overlap train∩val  = {len(overlap_tv)} (ожидаем 0)')


=== pasta_v4: split sizes {'train': 9415, 'val': 3138, 'test': 3138} ===


  brands: train=3037, val=1020, test=1022
  overlap train∩test = 0 (ожидаем 0)
  overlap train∩val  = 0 (ожидаем 0)

=== chocolate_v4: split sizes {'train': 8081, 'val': 2694, 'test': 2694} ===


  brands: train=2454, val=810, test=816
  overlap train∩test = 0 (ожидаем 0)
  overlap train∩val  = 0 (ожидаем 0)

=== cheeses_v4: split sizes {'train': 12724, 'val': 4242, 'test': 4242} ===


  brands: train=4157, val=1380, test=1362
  overlap train∩test = 0 (ожидаем 0)
  overlap train∩val  = 0 (ожидаем 0)


<a id='artifacts'></a>
## 8. Воспроизводимость и артефакты

Все ключевые артефакты сохраняются в виде parquet-файлов с детерминированным построением через скрипты (без интерактивного notebook'а в production-цепочке):

| Артефакт | Путь | Размер | Скрипт-сборщик |
|---|---|---|---|
| Raw OFF дамп | `~/off_work/food.parquet` (VM) | 7.14 ГБ | `hf_hub_download` (`src/data/download.py`) |
| Фильтрованные категории | `datasets/processed/{cat}_stratified_raw.parquet` | 45–100 МБ | `src/data/filter.py` (DuckDB SQL) |
| Silver labelling v6 | `datasets/processed/{cat}_stratified_silver_standard.parquet` | 0.5–1 МБ | `scripts/relabel_off_parallel.py` |
| Hybrid gold v3 (long) | `datasets/processed/{cat}_hybrid_gold_v3.parquet` | 1–2 МБ | `scripts/build_hybrid_gold_v3.py` |
| Gold v4 wide | `datasets/processed/{cat}_gold_v4_wide.parquet` | 1–2 МБ | `scripts/build_gold_v4_wide.py` |
| Brand-disjoint split | `datasets/processed/{cat}_gold_split.parquet` | 0.1 МБ | `scripts/build_brand_disjoint_split.py` |
| Embeddings cache | `datasets/processed/{cat}_stratified_embeddings.npy` | 18–22 МБ | `src/common.py:get_embeddings` (см. `02_training.ipynb`) |
| LLM-consensus золото | `datasets/processed/manual_gold_consensus.parquet` | ~50 КБ | `scripts/consensus_manual_gold.py` |

**Полный pipeline воспроизводится двумя командами (на VM, см. `CLAUDE.md` секцию Remote VM):**
```bash
# 1. Полная LLM-разметка (~30 мин, ~$40)
python scripts/relabel_off_parallel.py --category cheeses --workers 100 --out-suffix _v6

# 2. Сборка hybrid gold v3 и brand-disjoint split
python scripts/build_hybrid_gold_v3.py --category cheeses
python scripts/build_brand_disjoint_split.py --category cheeses --seed 42
```

Следующий шаг — `02_training.ipynb` (Layer 1 regex / Layer 2 ML / Layer 3 Bayes / Layer 0 router).